In [1]:
// load the formatting setup, the visualization functions, and the style sheet, install Z3 if necessary
#load "../setup.fsx"
#load "../visuals.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

# The Knight's Tour

This notebook computes a solution to the [knight's tour](https://en.wikipedia.org/wiki/Knight%27s_tour) using the constraint solver `Z3`.  

The function `row` takes one argument:
- `i` is an integer from the set $\{0, 1, \cdots, 63\}$.

It returns the name of the variable that specifies the *row* of the knight after its $i^{\textrm{th}}$ move.  Similarly, the function `col` returns the name of the variable that specifies the *column* of the knight after its $i^{\textrm{th}}$ move.

In [3]:
let row (i: int) : string = sprintf "R%d" i
let col (i: int) : string = sprintf "C%d" i

We represent rows and columns by *bit vectors* of length $4$.  A bit vector of length $4$ can represent the numbers from $-8$ to $7$.  Bit vectors are often handled more efficiently by `Z3` than integers.  The `.NET` interface of `Z3` does not provide operators for bit vectors.  Therefore, we define the following auxiliary functions:
- `bvAdd x d` takes a bit vector `x` and an integer `d` and returns the bit vector $x + d$,
- `bvNonNegative x` takes a bit vector `x` and returns a formula that is true iff `x` is not negative, when `x` is interpreted as a *signed* number.

In [4]:
let bvAdd (x: BitVecExpr) (d: int) : BitVecExpr =
    ctx.MkBVAdd(x, ctx.MkBV(d, x.SortSize))

let bvNonNegative (x: BitVecExpr) : BoolExpr =
    ctx.MkBVSGE(x, ctx.MkBV(0, x.SortSize))

The figure below shows the moves of a knight:  The knight on `e4` can jump to all red squares.

<img src="knight-moves.png" width="50%">

A knight moves two squares in one direction and one square in the other direction.  Hence the possible changes of the row and the column are the pairs $(\delta_r, \delta_c)$ with $\delta_r, \delta_c \in \{1, 2, -1, -2\}$ and $|\delta_r| \not= |\delta_c|$.

In [5]:
let S = [ 1; 2; -1; -2 ]
let DeltaSet = [ for x in S do for y in S do if abs x <> abs y then (x, y) ]
DeltaSet

[(1, 2); (1, -2); (2, 1); (2, -1); (-1, 2); (-1, -2); (-2, 1); (-2, -1)]

The function `isKnightMove` takes four arguments:
* `row` is a `Z3` variable that specifies the row of the position of the knight before the move.
* `col` is a `Z3` variable that specifies the column of the position of the knight before the move.
* `rowX` is a `Z3` variable that specifies the row of the position of the knight after the move.
* `colX` is a `Z3` variable that specifies the column of the position of the knight after the move.

It returns a formula that specifies that the move from `(row, col)` to `(rowX, colX)` is a legal move for a knight.
In order to form the *conjunction* of formulas we use the function `And`, 
while the *disjunction* is built with the function `Or`.  Both functions take a list of formulas.

In [6]:
let isKnightMove (row: BitVecExpr) (col: BitVecExpr) (rowX: BitVecExpr) (colX: BitVecExpr) : BoolExpr =
    Or [ for (dr, dc) in DeltaSet -> And [ rowX .= bvAdd row dr; colX .= bvAdd col dc ] ]

The function `allDifferent` takes two arguments:
* `Rows` is an array of `Z3` variables. The variable `Rows[i]` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `Cols` is an array of `Z3` variables. The variable `Cols[i]` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

The function returns a list of formulas stating that for $i \not= j$ the position after the $i^{\textrm{th}}$ move
differs from the position after the $j^{\textrm{th}}$ move.

In [7]:
let allDifferent (Rows: BitVecExpr[]) (Cols: BitVecExpr[]) : BoolExpr list =
    [ for i in 0 .. 62 do
        for j in i + 1 .. 63 ->
            Or [ Rows[i] .<> Rows[j]; Cols[i] .<> Cols[j] ] ]

The function `allConstraints` takes the same two arguments `Rows` and `Cols`.  It returns a list containing all constraints of the problem:
- all positions are different,
- the knight starts in the upper left corner,
- every move is a knight's move,
- all rows and columns are non-negative.  As every move is a knight's move, a position outside of the board can only be reached by a row or column that is negative or greater than $7$.  A bit vector of length $4$ that is non-negative is at most $7$.

In [8]:
let allConstraints (Rows: BitVecExpr[]) (Cols: BitVecExpr[]) : BoolExpr list =
    allDifferent Rows Cols
    @ [ Rows[0] .= BitVecVal 0L 4; Cols[0] .= BitVecVal 0L 4 ]
    @ [ for i in 0 .. 62 -> isKnightMove Rows[i] Cols[i] Rows[i+1] Cols[i+1] ]
    @ [ for i in 0 .. 63 do
          yield bvNonNegative Rows[i]
          yield bvNonNegative Cols[i] ]

The function `solve` takes no arguments.  It computes a solution of the knight's problem and returns this solution as a map that assigns values to the variables `Ri` and `Ci`.  If no solution is found, a message is printed and the empty map is returned.  Note that this function hides the function `solve` from `z3.fsx`.

In [9]:
let solve () : Map<string, int> =
    let Rows = [| for i in 0 .. 63 -> BitVec (row i) 4 |]
    let Cols = [| for i in 0 .. 63 -> BitVec (col i) 4 |]
    let Constraints = allConstraints Rows Cols
    let S = newSolver ()
    S.Add(Array.ofList Constraints)
    match S.Check() with
    | Status.SATISFIABLE ->
        let Model = S.Model
        Map ([ for i in 0 .. 63 -> (row i, intValue Model Rows[i]) ]
             @ [ for i in 0 .. 63 -> (col i, intValue Model Cols[i]) ])
    | Status.UNSATISFIABLE ->
        printfn "The problem is not solvable."
        Map.empty
    | _ ->
        printfn "Z3 cannot determine whether the problem is solvable."
        Map.empty

Unfortunately, the execution time of the following cell varies greatly between
different runs.  It might take from less than a minute up to half an hour.

In [10]:
#!time
let Solution = solve ()
Solution

map [("C0", 0); ("C1", 2); ("C10", 4); ("C11", 5); ("C12", 7); ("C13", 6); ("C14", 5); ("C15", 7); ("C16", 6); ("C17", 4); ("C18", 3); ("C19", 1); ("C2", 4); ("C20", 0); ("C21", 2); ("C22", 0); ("C23", 1); ("C24", 2); ("C25", 3); ("C26", 5); ("C27", 7); ("C28", 5); ("C29", 7); ("C3", 6); ("C30", 6); ("C31", 4); ("C32", 6); ("C33", 5); ("C34", 3); ("C35", 1); ("C36", 2); ("C37", 3); ("C38", 2); ("C39", 4); ("C4", 7); ("C40", 5); ("C41", 7); ("C42", 6); ("C43", 4); ("C44", 3); ("C45", 1); ("C46", 0); ("C47", 2); ("C48", 4); ("C49", 6); ("C5", 5); ("C50", 7); ("C51", 6); ("C52", 7); ("C53", 5); ("C54", 4); ("C55", 3); ("C56", 1); ("C57", 0); ("C58", 1); ("C59", 0); ("C6", 3); ("C60", 2); ("C61", 0); ("C62", 1); ("C63", 0); ("C7", 2); ("C8", 1); ("C9", 3); ("R0", 0); ("R1", 1); ("R10", 4); ("R11", 6); ("R12", 7); ("R13", 5); ("R14", 7); ("R15", 6); ("R16", 4); ("R17", 5); ("R18", 7); ("R19", 6); ("R2", 0); ("R20", 4); ("R21", 3); ("R22", 2); ("R23", 0); ("R24", 2); ("R25", 4); ("R26", 3); 

Wall time: 143042.3563ms

The function `createBoard` takes one argument:
- `Solution` is a map assigning values to the variables `Ri` and `Ci`.

It returns a matrix `Board` of size $8\times 8$, i.e. an array of arrays.  The following holds:
$$ \texttt{Board}[\texttt{R}i][\texttt{C}i] = i $$
Therefore, if `Board[r][c] = i`, then after the $i^{\textrm{th}}$ move the knight is located in row `r` and column `c`. 

In [11]:
let createBoard (Solution: Map<string, int>) : int[][] =
    let Board = Array.init 8 (fun _ -> Array.zeroCreate 8)
    for i in 1 .. 63 do
        let r = Solution[row i]
        let c = Solution[col i]
        Board[r][c] <- i
    Board

In [12]:
createBoard Solution

[|[|0; 23; 36; 55; 2; 53; 32; 29|]; [|63; 56; 1; 34; 31; 28; 3; 52|]; [|22; 35; 24; 37; 54; 33; 30; 27|]; [|57; 62; 21; 6; 39; 26; 51; 4|]; [|20; 45; 38; 25; 10; 5; 16; 41|]; [|61; 58; 7; 44; 17; 40; 13; 50|]; [|46; 19; 60; 9; 48; 11; 42; 15|]; [|59; 8; 47; 18; 43; 14; 49; 12|]|]

The function `printBoard` takes one argument:
- `Board` is a square matrix of integers.

It prints the given `Board`.  Every entry is padded to the width of the widest entry.  The function `String.replicate k s` returns the string `s` repeated `k` times.

In [13]:
let printBoard (Board: int[][]) : unit =
    let n = Board.Length
    // Determine the width of the widest element in the matrix
    let width = Board |> Array.collect id |> Array.map (fun e -> (string e).Length) |> Array.max
    let line (left: string) (middle: string) (right: string) =
        left + String.concat middle (List.replicate n (String.replicate (width + 2) "═")) + right
    printfn "%s" (line "╔" "╦" "╗")
    for i in 0 .. n - 1 do
        printfn "║%s" (String.concat "" [ for e in Board[i] -> sprintf " %*d ║" width e ])
        if i < n - 1 then
            printfn "%s" (line "╠" "╬" "╣")
    printfn "%s" (line "╚" "╩" "╝")

In [14]:
printBoard (createBoard Solution)

╔════╦════╦════╦════╦════╦════╦════╦════╗

║

  0 ║ 23 ║ 36 ║ 55 ║  2 ║ 53 ║ 32 ║ 29 ║

╠════╬════╬════╬════╬════╬════╬════╬════╣

║

...

 46 ║ 19 ║ 60 ║  9 ║ 48 ║ 11 ║ 42 ║ 15 ║

╠════╬════╬════╬════╬════╬════╬════╬════╣

║

 59 ║  8 ║ 47 ║ 18 ║ 43 ║ 14 ║ 49 ║ 12 ║

╚════╩════╩════╩════╩════╩════╩════╩════╝

# Visualization

The function `showSolution` takes two arguments:
- `Board` is the matrix computed by `createBoard`.  We have `Board[row][col] = k` if the $k^\textrm{th}$ move leads the knight to the position `(row, col)`,
- `width` is the width of the image as a CSS length.

It displays the path of the knight on a chess board.  The function `knightsTour` is defined in the file `visuals.fsx`.

**Implementation:** The map `Positions` assigns to every number $k$ the position reached by the $k^\textrm{th}$ move.  Then the path is the list of these positions ordered by $k$.

In [15]:
let showSolution (Board: int[][]) (width: string) =
    let n = Board.Length
    let Positions = Map [ for row in 0 .. n - 1 do
                            for col in 0 .. n - 1 -> (Board[row][col], (row, col)) ]
    let Path = [ for k in 0 .. n * n - 1 -> Positions[k] ]
    knightsTour n Path width

In [16]:
showSolution (createBoard Solution) "50%"

<polyline points="25,25 125,75 225,25 325,75 375,175 275,225 175,175 125,275 75,375 175,325 225,225 275,325 375,375 325,275 275,375 375,325 325,225 225,275 175,375 75,325 25,225 125,175 25,125 75,25 125,125 175,225 275,175 375,125 275,75 375,25 325,125 225,75 325,25 275,125 175,75 75,125 125,25 175,125 125,225 225,175 275,275 375,225 325,325 225,375 175,275 75,225 25,325 125,375 225,325 325,375 375,275 325,175 375,75 275,25 225,125 175,25 75,75 25,175 75,275 25,375 125,325 25,275 75,175 25,75" fill="none" stroke="darkblue" stroke-width="2" opacity="0.7"/> 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63